# AeroRAG — Evaluation Notebook

Tests the full RAG pipeline with 10 questions across 3 categories:
- **Exact Fact Retrieval** (4 questions) — single-source, verifiable facts
- **Multi-Chunk Synthesis** (3 questions) — require combining multiple sources
- **Out-of-Scope Refusal** (3 questions) — must return the missing-info response

Human-judged pass/fail. Results saved to `docs/evaluation_results.md`.

In [2]:
import sys
import os
import time

# Add src/ to path so we can import project modules
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
sys.path.insert(0, '../src')

from generator import generate

print('✅ Imports successful')

✅ Imports successful


In [3]:
# ── Test Question Bank ─────────────────────────────────────────────────────

EXACT_FACT_QUESTIONS = [
    'Who were the Wright Brothers and when did they achieve powered flight?',
    'What year did Yuri Gagarin become the first human in space?',
    'What caused the Space Shuttle Challenger disaster?',
    'What is the International Space Station and who operates it?',
]

MULTI_CHUNK_QUESTIONS = [
    'Compare the Apollo 11 and Apollo 13 missions — what were their outcomes?',
    'How did SpaceX change rocket technology with the Falcon 9?',
    'What were the key differences between the Mercury and Gemini programs?',
]

OUT_OF_SCOPE_QUESTIONS = [
    'Who won the 2024 FIFA World Cup?',
    'What is the population of Karachi?',
    'What is the current price of Bitcoin?',
]

ALL_TESTS = (
    [(q, 'Exact Fact')    for q in EXACT_FACT_QUESTIONS] +
    [(q, 'Multi-Chunk')   for q in MULTI_CHUNK_QUESTIONS] +
    [(q, 'Out-of-Scope')  for q in OUT_OF_SCOPE_QUESTIONS]
)

print(f'Total test questions: {len(ALL_TESTS)}')

Total test questions: 10


In [4]:
# ── Run All Tests ──────────────────────────────────────────────────────────

MISSING_INFO_MARKER = 'missing from the provided dataset'
results = []

for i, (question, category) in enumerate(ALL_TESTS, 1):
    print(f'\n[{i:02d}/{len(ALL_TESTS)}] [{category}]')
    print(f'Q: {question}')
    
    t0 = time.time()
    answer, sources = generate(question)
    elapsed = round(time.time() - t0, 2)
    
    refused = MISSING_INFO_MARKER.lower() in answer.lower()
    
    print(f'A: {answer[:300]}{"..." if len(answer) > 300 else ""}')
    print(f'Sources: {len(sources)} | Time: {elapsed}s | Refused: {refused}')
    
    results.append({
        'id':       i,
        'category': category,
        'question': question,
        'answer':   answer,
        'sources':  len(sources),
        'refused':  refused,
        'time_s':   elapsed,
        'pass':     None,  # human-judged below
    })
    time.sleep(25)

print('\n✅ All tests complete')

INFO — Loading embedding model for retrieval: all-MiniLM-L6-v2



[01/10] [Exact Fact]
Q: Who were the Wright Brothers and when did they achieve powered flight?


INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
WARNING — Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO — Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO — HTTP Request: HEAD https://huggingface.co/sentence-

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/1.1 200 OK"
INFO — HTTP Request: HEAD https://huggingf

...


[08/10] [Out-of-Scope]
Q: Who won the 2024 FIFA World Cup?
A: The requested target info is missing from the provided dataset.
Sources: 15 | Time: 0.85s | Refused: True


INFO — Query: 'What is the population of Karachi?' → 15 chunks above threshold 0.15
INFO — Generating response | query='What is the population of Karachi?' | context_chunks=15



[09/10] [Out-of-Scope]
Q: What is the population of Karachi?
A: The requested target info is missing from the provided dataset.
Sources: 15 | Time: 1.54s | Refused: True

[10/10] [Out-of-Scope]
Q: What is the current price of Bitcoin?


INFO — Query: 'What is the current price of Bitcoin?' → 15 chunks above threshold 0.15
INFO — Generating response | query='What is the current price of Bitcoin?' | context_chunks=15


A: The requested target info is missing from the provided dataset.
Sources: 15 | Time: 3.85s | Refused: True

✅ All tests complete


In [5]:
# ── Human Grading ──────────────────────────────────────────────────────────
# Fill in True/False manually after reviewing each answer above.

human_grades = [
    # (test_id, pass_bool)     ← edit these after reviewing outputs
    (1,  True),   # Wright Brothers
    (2,  True),   # Yuri Gagarin
    (3,  True),   # Challenger disaster
    (4,  True),   # ISS
    (5,  True),   # Apollo 11 vs 13
    (6,  True),   # SpaceX Falcon 9
    (7,  True),   # Mercury vs Gemini
    (8,  True),   # FIFA World Cup (should refuse)
    (9,  True),   # Karachi population (should refuse)
    (10, True),   # Bitcoin price (should refuse)
]

grade_map = {tid: p for tid, p in human_grades}
for r in results:
    r['pass'] = grade_map.get(r['id'], None)

print('Grades applied.')

Grades applied.


In [6]:
# ── Summary Table ──────────────────────────────────────────────────────────

import pandas as pd

df = pd.DataFrame(results)
df['pass_label'] = df['pass'].map({True: '✅ Pass', False: '❌ Fail', None: '⏳ Pending'})

display_cols = ['id', 'category', 'question', 'sources', 'refused', 'time_s', 'pass_label']
print(df[display_cols].to_string(index=False))

total   = len(df)
passing = df['pass'].sum()
avg_t   = df['time_s'].mean()

# Out-of-scope: correct if model refused
oos_df      = df[df['category'] == 'Out-of-Scope']
hallucinated = (oos_df['refused'] == False).sum()

print(f'\n--- Summary ---')
print(f'Pass rate          : {passing}/{total} ({100*passing/total:.0f}%)')
print(f'Avg response time  : {avg_t:.1f}s')
print(f'Hallucinations     : {hallucinated}/{len(oos_df)} out-of-scope questions answered incorrectly')

 id     category                                                                 question  sources  refused  time_s pass_label
  1   Exact Fact   Who were the Wright Brothers and when did they achieve powered flight?       15    False   15.72     ✅ Pass
  2   Exact Fact              What year did Yuri Gagarin become the first human in space?       15    False    0.94     ✅ Pass
  3   Exact Fact                       What caused the Space Shuttle Challenger disaster?       15    False    0.95     ✅ Pass
  4   Exact Fact             What is the International Space Station and who operates it?       15    False    1.89     ✅ Pass
  5  Multi-Chunk Compare the Apollo 11 and Apollo 13 missions — what were their outcomes?       15    False    2.94     ✅ Pass
  6  Multi-Chunk               How did SpaceX change rocket technology with the Falcon 9?       15    False    1.36     ✅ Pass
  7  Multi-Chunk   What were the key differences between the Mercury and Gemini programs?       15    False    

In [7]:
# ── Save Results to docs/evaluation_results.md ────────────────────────────

import os

os.makedirs('../docs', exist_ok=True)
output_path = '../docs/evaluation_results.md'

lines = [
    '# AeroRAG — Evaluation Results\n',
    f'Pass rate: {passing}/{total} | Avg time: {avg_t:.1f}s | Hallucinations: {hallucinated}/{len(oos_df)}\n',
    '\n',
    '| # | Category | Question | Sources | Refused | Time(s) | Pass |\n',
    '|---|---|---|---|---|---|---|\n',
]

for r in results:
    lines.append(
        f"| {r['id']} | {r['category']} | {r['question'][:60]}... "
        f"| {r['sources']} | {r['refused']} | {r['time_s']}s "
        f"| {'✅' if r['pass'] else '❌'} |\n"
    )

lines.append('\n## Sample Answers\n\n')
for r in results[:4]:
    lines.append(f"### Q{r['id']}: {r['question']}\n")
    lines.append(f"{r['answer'][:500]}\n\n---\n\n")

with open(output_path, 'w', encoding='utf-8') as f:
    f.writelines(lines)

print(f'✅ Results saved to {output_path}')

✅ Results saved to ../docs/evaluation_results.md
